In [1]:
#r "nuget: CsvHelper, 33.0.1"
#r "nuget: Accord.MachineLearning, 3.8.0"
#r "nuget: Plotly.NET, 5.0.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "nuget: Plotly.NET.CSharp, 0.13.0"
using System;
using System.Collections.Generic;
using System.IO;
using System.Linq;
using System.Globalization;
using System.Net.Http; 
using CsvHelper;
using CsvHelper.Configuration;
using CsvHelper.Configuration.Attributes;
using Accord.MachineLearning.Rules;
using System.IO.Compression; 
using Plotly.NET;
using Plotly.NET.LayoutObjects;
using Plotly.NET.Interactive;
using Plotly.NET.CSharp;
using Chart = Plotly.NET.CSharp.Chart;

string URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip";
HttpClient client = new HttpClient();
Stream stream = await client.GetStreamAsync(URL);

//1
public class Movie
{
    [Name("movieId")]
    public int MovieId { get; set; }
    [Name("title")]
    public string Title { get; set; }
    [Name("genres")]
    public string Genres { get; set; }
}
public class Rating
{
    [Name("userId")]
    public int UserId { get; set; }
    [Name("movieId")]
    public int MovieId { get; set; }
    [Name("rating")]
    public double Score { get; set; } 
    [Name("timestamp")]
    public long Timestamp { get; set; }
}
List<Movie> movies = new List<Movie>();
List<Rating> ratings = new List<Rating>();
using (ZipArchive archive = new ZipArchive(stream, ZipArchiveMode.Read))
{
    var moviesEntry = archive.Entries.FirstOrDefault(e => e.FullName.EndsWith("movies.csv"));
    if (moviesEntry != null)
    {
        using (Stream entryStream = moviesEntry.Open())
        using (StreamReader reader = new StreamReader(entryStream))
        using (CsvReader csv = new CsvReader(reader, CultureInfo.InvariantCulture))
        {
            movies = csv.GetRecords<Movie>().ToList();
        }
    }

    var ratingsEntry = archive.Entries.FirstOrDefault(e => e.FullName.EndsWith("ratings.csv"));
    if (ratingsEntry != null)
    {
        using (Stream entryStream = ratingsEntry.Open())
        using (StreamReader reader = new StreamReader(entryStream))
        using (CsvReader csv = new CsvReader(reader, CultureInfo.InvariantCulture))
        {
            ratings = csv.GetRecords<Rating>().ToList();
        }
    }
}
var mergedData = ratings.Join(movies,
    rating => rating.MovieId,  
    movie => movie.MovieId,    
    (rating, movie) => new    
    { 
        UserId = rating.UserId, 
        MovieTitle = movie.Title, 
        Score = rating.Score 
    }).ToList();

int uniqueUsersCount = mergedData.Select(m => m.UserId).Distinct().Count();
int uniqueMoviesCount = mergedData.Select(m => m.MovieTitle).Distinct().Count();
Console.WriteLine($"Кількість фільмів: {movies.Count}");
Console.WriteLine($"Кількість користувачів: {ratings.Count}");
Console.WriteLine($"Розмір об'єднаної таблиці (кількість позитивних оцінок): {mergedData.Count}");
Console.WriteLine($"Кількість унікальних користувачів: {uniqueUsersCount}");
Console.WriteLine($"Кількість унікальних фільмів: {uniqueMoviesCount}");
List<int> uniqueUserIds = mergedData.Select(m => m.UserId).Distinct().ToList();
List<string> uniqueMovieTitles = mergedData.Select(m => m.MovieTitle).Distinct().ToList();
//2
int[,] matrixMoveRating = new int[uniqueUsersCount, uniqueMoviesCount];
foreach (var data in mergedData.Where(d => d.Score >= 4.0))
{
    int rowIndex = uniqueUserIds.IndexOf(data.UserId);
    int colIndex = uniqueMovieTitles.IndexOf(data.MovieTitle);
    matrixMoveRating[rowIndex, colIndex] = 1;
}
int rowsToDisplay = 10; 
int colsToDisplay = 5;  
Console.Write($"{"User ID",-10} | ");
int[] colWidths = new int[colsToDisplay];
for (int j = 0; j < colsToDisplay; j++)
{
    string title = uniqueMovieTitles[j];
    colWidths[j] = title.Length + 3;
    Console.Write($"{title.PadRight(colWidths[j])}");
}
Console.WriteLine();
int totalWidth = 13 + colWidths.Sum();
Console.WriteLine(new string('-', totalWidth));
for (int i = 0; i < rowsToDisplay; i++)
{
    Console.Write($"{uniqueUserIds[i],-10} | "); 
    for (int j = 0; j < colsToDisplay; j++)
    {
        Console.Write($"{matrixMoveRating[i, j].ToString().PadRight(colWidths[j])}");
    }
    Console.WriteLine();
}

//3

List<List<int>> transactions = new List<List<int>>();
for (int i = 0; i<uniqueUsersCount;i++){
    List<int> tmp = new List<int>();
    for (int j = 0; j<uniqueMoviesCount;j++){
        if(matrixMoveRating[i,j] == 1){
            tmp.Add(j);
        }
    }
    transactions.Add(tmp);
}
public void RunApriori(float min_support, List<List<int>> data) 
{
    Apriori apriori = new Apriori(threshold: (int)(data.Count * min_support), confidence: 0);
    var classifier = apriori.Learn(data.Select(d => d.ToArray()).ToArray());

    var pairs = classifier.Rules
        .Select(r => new {
            Set = r.X.Concat(r.Y).OrderBy(i => i).ToArray(), 
            Support = r.Support 
        })
        .Where(r => r.Set.Length == 2) 
        .GroupBy(r => string.Join(",", r.Set)) 
        .Select(g => g.First())
        .ToList();

    var top10Pairs = pairs
        .OrderByDescending(p => p.Support)
        .Take(10) 
        .ToList();
    var frequentSets = classifier.Rules.Select(r => r.X.Concat(r.Y).Distinct().ToArray()).ToList();
    int pairsCount = frequentSets.Count(set => set.Length == 2);
    Console.WriteLine($"Поріг support {min_support:P0}:");
    Console.WriteLine($" - Знайдено унікальних комбінацій (правил): {classifier.Rules.Count()}");
    Console.WriteLine($" - З них ПАР: {pairsCount}");
    Console.WriteLine($"--- ТОП-10 ПАР для порогу {min_support:P0} ---");
    foreach (var p in top10Pairs)
    {
        string movie1 = uniqueMovieTitles[p.Set[0]];
        string movie2 = uniqueMovieTitles[p.Set[1]];
        double sPercent = (double)p.Support / data.Count;

        Console.WriteLine($"{sPercent:P1} | {movie1} + {movie2}");
    }
    Console.WriteLine("-----------------------------------");
}
RunApriori(0.06f, transactions);
RunApriori(0.07f, transactions);
RunApriori(0.08f, transactions);
RunApriori(0.09f, transactions);
RunApriori(0.1f, transactions);
RunApriori(0.2f, transactions);
RunApriori(0.3f, transactions);

//4
int totalTransactions = transactions.Count;
Apriori apriori = new Apriori(threshold: (int)(totalTransactions * 0.1), confidence: 0.1);
var classifier = apriori.Learn(transactions.Select(d => d.ToArray()).ToArray());

var rules = classifier.Rules.ToList();

var enrichedRules = rules.Select(r => {
    double ruleSupport = (double)r.Support / totalTransactions;
    int supportYCount = transactions.Count(t => r.Y.All(y => t.Contains(y)));
    double supportY = (double)supportYCount / totalTransactions;
    double lift = supportY > 0 ? r.Confidence / supportY : 0;

    return new {
        X = r.X,
        Y = r.Y,
        Support = ruleSupport,
        Confidence = r.Confidence,
        Lift = lift
    };
}).ToList();

var top10Lift = enrichedRules.OrderByDescending(r => r.Lift).Take(10).ToList();

Console.WriteLine("--- ТОП-10 ПРАВИЛ ЗА LIFT ---");
foreach (var r in top10Lift)
{
    string xNames = string.Join(" + ", r.X.Select(i => uniqueMovieTitles[i]));
    string yNames = string.Join(" + ", r.Y.Select(i => uniqueMovieTitles[i]));
    Console.WriteLine($"{xNames} => {yNames} | Support: {r.Support:P1}, Conf: {r.Confidence:P1}, Lift: {r.Lift:F2}");
}

var supportVals = enrichedRules.Select(r => r.Support).ToArray();
var confVals = enrichedRules.Select(r => r.Confidence).ToArray();
var liftVals = enrichedRules.Select(r => r.Lift).ToArray();

var chart1 = Chart.Scatter<double, double, string>(
    x: supportVals,
    y: confVals,
    mode: StyleParam.Mode.Markers
)
.WithXAxisStyle(Title.init("Support"))
.WithYAxisStyle(Title.init("Confidence"))
.WithTitle("Support vs Confidence");

var chart2 = Chart.Scatter<double, double, string>(
    x: liftVals,
    y: confVals,
    mode: StyleParam.Mode.Markers
)
.WithXAxisStyle(Title.init("Lift"))
.WithYAxisStyle(Title.init("Confidence"))
.WithTitle("Lift vs Confidence");

Plotly.NET.CSharp.GenericChartExtensions.Show(chart1);
Plotly.NET.CSharp.GenericChartExtensions.Show(chart2);


Installed Packages Accord.MachineLearning, 3.8.0 CsvHelper, 33.0.1 Plotly.NET, 5.0.0 Plotly.NET.CSharp, 0.13.0 Plotly.NET.Interactive, 5.0.0

Loading extensions from `C:\Users\Діма\.nuget\packages\plotly.net.interactive\5.0.0\lib\netstandard2.1\Plotly.NET.Interactive.dll`

Кількість фільмів: 9742
Кількість користувачів: 100836
Розмір об'єднаної таблиці (кількість позитивних оцінок): 100836
Кількість унікальних користувачів: 610
Кількість унікальних фільмів: 9719
User ID    | Toy Story (1995)   Grumpier Old Men (1995)   Heat (1995)   Seven (a.k.a. Se7en) (1995)   Usual Suspects, The (1995)   
-----------------------------------------------------------------------------------------------------------------------------------
1          | 1                  1                         1             1                             1                            
2          | 0                  0                         0             0                             0                            
3          | 0                  0                         0             0                             0                            
4          | 0                  0                         0             0                             0                            
5          | 1 

Error: Command cancelled.